In [1]:
import sys
sys.path.append("../")

import sympy as sp
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_point, compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()

ref: [Affine Body Prismatic Joint](https://spirimirror.github.io/libuipc-doc/specification/constitutions/affine_body_prismatic_joint/)

![Affine Body Prismatic Joint](../../docs/specification/constitutions/media/affine_body_prismatic_joint_fig1.drawio.svg)

$$
C_0 = (\mathbf{c}_j - \mathbf{c}_i) \cdot \hat{\mathbf{t}}_i  - \tilde{d} = \mathbf{0}
$$

$$
C_1 = -(\mathbf{c}_i - \mathbf{c}_j) \cdot (\hat{\mathbf{t}}_j) - \tilde{d} = \mathbf{0}
$$

$$
\mathbf{F}_{01} = \begin{bmatrix}
\mathbf{c}_j - \mathbf{c}_i \\
\hat{\mathbf{t}}_i \\
-\hat{\mathbf{t}}_j \\
\end{bmatrix}_{9 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{01} = \begin{bmatrix}
-\mathbf{J}(\bar{\mathbf{c}}_i) & \mathbf{J}(\bar{\mathbf{c}}_j) \\
\mathring{\mathbf{J}}(\bar{\mathbf{t}}_i) & \mathbf{0} \\
\mathbf{0} & -\mathring{\mathbf{J}}(\bar{\mathbf{t}}_j)
\end{bmatrix}_{9 \times 24}
$$

$$
\mathbf{F}_{01} = \mathbf{J}_{01} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [2]:
kappa  = Eigen.Scalar("kappa")

# Basis vectors
Ci_bar = Eigen.Vector("Ci_bar",3)
Ti_bar = Eigen.Vector("Ti_bar",3)

Cj_bar = Eigen.Vector("Cj_bar",3)
Tj_bar = Eigen.Vector("Tj_bar",3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi",12)
qj = Eigen.Vector("qj",12)

In [3]:
# Mapping
J01 = sp.Matrix.zeros(9,24)
J01[0:3, 0:12] = -compute_J_point(Ci_bar)
J01[0:3, 12:24] = compute_J_point(Cj_bar)
J01[3:6, 0:12] = compute_J_vec(Ti_bar)
J01[6:9, 12:24] = -compute_J_vec(Tj_bar)

content = ""

# from ABD q to F
F0_q = J01 @ sp.Matrix.vstack(qi, qj)
Cl_F01 = Gen.Closure(Ci_bar, Ti_bar, qi, # Affine Body i
                     Cj_bar, Tj_bar, qj) # Affine Body j
# from F Gradient to ABD Gradient
G9 = Eigen.Vector("G9",9)
J01T_G01 = J01.T @ G9
Cl_G01 = Gen.Closure(G9, 
                     Ci_bar, Ti_bar, # Affine Body i
                     Cj_bar, Tj_bar) # Affine Body j
# from F Hessian to ABD Hessian
H9x9 = Eigen.Matrix("H9x9",9,9)
J01T_H01_J01 = J01.T @ H9x9 @ J01
Cl_H01 = Gen.Closure(H9x9, 
                     Ci_bar, Ti_bar, # Affine Body i
                     Cj_bar, Tj_bar) # Affine Body j

content += f""" 
// Prismatic Joint: C0 C1
// Mapping between ABD qi qj to F01

{Cl_F01("F01_q", F0_q)}
{Cl_G01("J01T_G01", J01T_G01)}
{Cl_H01("J01T_H01_J01", J01T_H01_J01)}

"""

F01_q = Eigen.Vector("F01_q", 9)
dij = F01_q[0:3, 0]
ti = F01_q[3:6, 0]
tj = F01_q[6:9, 0]

d_tidle = Eigen.Scalar("d_tidle")[0]

distance_ij = dij.dot(ti)
distance_ji = (-dij).dot(tj)

distance = Eigen.Scalar("distance")
Cl_Distance = Gen.Closure(F01_q)
content += f""" 
// Distance between two bodies along the prismatic joint(average of two directions)
{Cl_Distance("Distance", 0.5 * (distance_ij+ distance_ji))}

"""

E01 = 0.5 * kappa * ((distance_ij - d_tidle)**2 + (distance_ji - d_tidle)**2)
Cl_E01 = Gen.Closure(kappa, F01_q, d_tidle)


dE01dF01 = VecDiff(E01, F01_q)
ddE01ddF01 = VecDiff(dE01dF01, F01_q)
content += f""" // Prismatic Joint Energy and Derivatives

{Cl_E01("E01", E01)}
{Cl_E01("dE01dF01", dE01dF01)}
{Cl_E01("ddE01ddF01", ddE01ddF01)}

// -------------------------------------------------------------------------
"""

In [4]:
f = open(backend_source_dir('cuda') / 'affine_body/constitutions/sym/affine_body_driving_prismatic_joint.inl', 'w')
f.write(content)
f.close()